# Fase 2 — Finalização da máscara de referência

Finalização da máscara binária de café para a Região Geográfica Imediata de Guaxupé - MG (código IBGE 310044), a partir da fonte de verdade de campo escolhida na comparação.

Este estágio lê a fonte selecionada em `masks.selected_source`, alinha a máscara ao grid da composição Sentinel-2, aplica a binarização quando a fonte for o k-means AlphaEarth e grava a máscara final como GeoTIFF no diretório processado.

## Detecção da raiz do repositório

Localiza a raiz do repositório a partir do diretório corrente e a insere no caminho de importação, garantindo o acesso ao pacote `src/`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


# Sobe os diretórios até encontrar src/config.yaml, marcador da raiz do projeto.
def _find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "config.yaml").is_file():
            return candidate
    raise RuntimeError("Raiz do repositório não localizada (src/config.yaml ausente).")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Raiz do projeto: {PROJECT_ROOT}")

## Detecção da plataforma

Identifica o ambiente de execução (Kaggle, Colab ou local) para adaptar a instalação de dependências e a leitura de segredos.

In [ ]:
import importlib.util
import os


# Heurística por variáveis de ambiente e presença de diretórios característicos.
def detect_platform() -> str:
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").is_dir():
        return "kaggle"
    if "COLAB_GPU" in os.environ or importlib.util.find_spec("google.colab") is not None:
        return "colab"
    return "local"


PLATFORM = detect_platform()
print(f"Plataforma detectada: {PLATFORM}")

## Instalação condicional das dependências

Em Kaggle/Colab instala o pacote com os extras geoespaciais e de aprendizado de máquina. No ambiente local a instalação é ignorada, pois é gerenciada por `uv` e pelo CI.

In [ ]:
import subprocess

# Instala o projeto editavelmente com os extras necessários apenas em nuvem.
if PLATFORM in {"kaggle", "colab"}:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[geo,ml]"],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print("Dependências instaladas.")
else:
    print("Ambiente local: instalação ignorada (gerenciada por uv/CI).")

## Carregamento da configuração única

Lê a configuração de `src/config.yaml` por meio de `src/config.py`, fonte única de verdade de caminhos, bandas, parâmetros e sementes.

In [ ]:
from src.config import CONFIG

# Exibe os parâmetros de máscaras definidos na configuração.
print(f"Fontes candidatas: {CONFIG.get('masks.candidates')}")
print(f"Fonte selecionada: {CONFIG.get('masks.selected_source')}")
print(f"Diretório final: {CONFIG.get('masks.final_dir')}")

## Fixação das sementes

Fixa as sementes de `python`, `numpy`, `torch` e `cuda` para garantir a reprodutibilidade das etapas amostrais (ex.: clusterização).

In [ ]:
from src.config import seed_everything

# Aplica a semente global definida na configuração.
resolved_seed = seed_everything()
print(f"Sementes fixadas em {resolved_seed}.")

## Carregamento de segredos

Em Kaggle/Colab injeta os segredos do cofre da plataforma nas variáveis de ambiente esperadas pelo pacote. Nenhum valor é impresso. No ambiente local, os segredos devem vir de variáveis de ambiente ou do arquivo `.env`.

In [ ]:
# Nomes das variáveis de ambiente consumidas pela aquisição.
SECRET_NAMES = (
    "GEE_SERVICE_ACCOUNT_EMAIL",
    "GEE_PROJECT",
    "GEE_SERVICE_ACCOUNT_KEY_JSON",
)

if PLATFORM == "kaggle":
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()
    for name in SECRET_NAMES:
        try:
            os.environ[name] = client.get_secret(name)
        except Exception:
            print(f"Segredo ausente no Kaggle: {name}")
elif PLATFORM == "colab":
    from google.colab import userdata

    for name in SECRET_NAMES:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            print(f"Segredo ausente no Colab: {name}")
else:
    print("Ambiente local: segredos esperados via variáveis de ambiente/.env.")

## Autenticação no Google Earth Engine

Inicializa o Earth Engine com a conta de serviço lida exclusivamente do ambiente e interrompe a execução caso as credenciais estejam ausentes, pois toda a fase depende do serviço.

In [ ]:
from src.data.gee_client import init_ee

# Inicializa o cliente do Earth Engine; sem credenciais a fase não prossegue.
ee = init_ee()
print("Earth Engine autenticado com sucesso.")

## Carregamento da área de estudo

Carrega a malha vetorial do IBGE e recorta o registro da Região Geográfica Imediata de Guaxupé, convertendo a geometria para o formato do Earth Engine.

In [ ]:
from src.data.aoi import geometry_bounds, geometry_to_ee, get_region_geometry

# Obtém a geometria unificada da região e a converte para ee.Geometry.
aoi_geometry = get_region_geometry()
aoi_ee = geometry_to_ee(ee, aoi_geometry)
print(f"Limites (minx, miny, maxx, maxy): {geometry_bounds(aoi_geometry)}")

## Seleção da fonte de verdade de campo

Lê a fonte eleita na comparação (`masks.selected_source`) e valida sua presença entre as candidatas definidas na configuração.

In [ ]:
# Valida a fonte escolhida em relação às candidatas definidas na configuração.
candidates = list(CONFIG.get("masks.candidates", []))
selected_source = CONFIG.get("masks.selected_source")
print(f"Fonte selecionada: {selected_source}")
assert selected_source in candidates, f"Fonte inválida: {selected_source}"

## Resolução dos arquivos de entrada

Deriva da configuração os caminhos da composição de referência e do GeoTIFF da fonte selecionada, preparando o diretório intermediário de alinhamento.

In [ ]:
from src.data.gee_client import make_export_description
from src.data.mask_compare import mask_candidate_filename

# Deriva caminhos esperados a partir da configuração (sem caminhos codificados).
prefix = CONFIG.get("gee.export_prefix")
region_code = CONFIG.get("aoi.region_code")
start_date = CONFIG.get("gee.start_date")
end_date = CONFIG.get("gee.end_date")
local_dir = CONFIG.paths.root / str(CONFIG.get("masks.local_dir"))
interim_dir = CONFIG.paths.interim / "masks"
interim_dir.mkdir(parents=True, exist_ok=True)

composite_file = local_dir / f"{make_export_description(prefix, region_code, start_date, end_date)}.tif"
source_file = local_dir / mask_candidate_filename(selected_source, prefix, region_code, start_date, end_date)
print(f"Composição: {composite_file.name} | presente={composite_file.is_file()}")
print(f"Fonte selecionada: {source_file.name} | presente={source_file.is_file()}")

## Alinhamento da fonte selecionada

Reprojeta a máscara da fonte eleita para o grid exato da composição Sentinel-2 e carrega o arranjo resultante.

In [ ]:
import numpy as np

from src.data.raster_io import align_to_reference, read_band

# Alinha a fonte eleita ao grid da composição.
aligned_file = interim_dir / f"{selected_source}_aligned.tif"
align_to_reference(source_file, composite_file, aligned_file)
aligned = read_band(aligned_file)
print(f"Máscara alinhada: {aligned.shape} | valores={np.unique(aligned)}")

## Binarização final da máscara de clusters

Quando a fonte selecionada é o k-means AlphaEarth, converte o mapa de clusters em binário usando a prevalência de café na referência MapBiomas já alinhada.

In [ ]:
from src.data.mask_compare import cluster_prevalence, threshold_clusters

# Se a fonte for o k-means, converte para binária com base na referência MapBiomas.
if selected_source == "alphaearth_clusters":
    mapbiomas_file = interim_dir / "mapbiomas_coffee_aligned.tif"
    reference = read_band(mapbiomas_file)
    prevalence = cluster_prevalence(aligned, reference)
    print(f"Prevalência por cluster: {prevalence}")
    aligned = threshold_clusters(
        aligned, prevalence, float(CONFIG.get("masks.cluster_threshold", 0.5))
    )
print(f"Máscara binária final: {np.unique(aligned)}")

## Persistência da máscara final

Grava a máscara binária final como GeoTIFF de banda única no diretório processado, preservando o georreferenciamento do grid da composição.

In [ ]:
from src.data.raster_io import write_single_band

# Grava a máscara final como GeoTIFF de banda única no diretório processado.
final_dir = CONFIG.paths.root / str(CONFIG.get("masks.final_dir"))
final_dir.mkdir(parents=True, exist_ok=True)
final_file = final_dir / f"{selected_source}_final.tif"
write_single_band(aligned, final_file, reference_path=aligned_file)
print(f"Máscara final gravada: {final_file}")

## Relatório de cobertura de café

Calcula e exibe a proporção de pixels de café na máscara finalizada, insumo para a triagem de patches na fase de construção do conjunto de dados.

In [ ]:
from src.data.mask_compare import coffee_ratio

# Proporção de pixels de café na máscara finalizada.
ratio = coffee_ratio(aligned)
print(f"Proporção de café na área de estudo: {ratio:.4f}")